# Watch OpenAI Pricer

Price watch auction lots with GPT-4.1 Mini. Input paths are defined explicitly below so the loaded files are visible before running API calls.


## Configuration


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "watch_prompt_builder.py").exists():
    WATCH_DIR = NOTEBOOK_DIR
else:
    WATCH_DIR = Path("Base_OpenAI/watch").resolve()

BASE_OPENAI = WATCH_DIR.parent
WORKSPACE_ROOT = BASE_OPENAI.parent
sys.path.insert(0, str(BASE_OPENAI))
sys.path.insert(0, str(WATCH_DIR))

OUTPUT_DIR = WATCH_DIR / "outputs"
WATCH_DATA_DIR = WORKSPACE_ROOT / "Data" / "watch"
QUARTERS = ("2025Q1", "2025Q2", "2025Q3", "2025Q4")
QUARTER_FILES = {quarter: WATCH_DATA_DIR / f"watch_sales_{quarter}_dedup_lot_url.csv" for quarter in QUARTERS}

MODEL = "gpt-4.1-mini"
INCLUDE_AUCTION_ESTIMATES = False
INCLUDE_MARKET_INDICATORS = True
VARIANT = "with_estimates" if INCLUDE_AUCTION_ESTIMATES else "no_estimates"
TARGET_COL = "ah_price_usd"
ROWS_PER_QUARTER = None # Set to None for a full run.
MAX_ROWS = None
RANDOM_STATE = 42
RESUME = True
CHECKPOINT_EVERY = 10
MAX_WORKERS = 50
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 300
MAX_RETRIES = 3
ENV_FILE = BASE_OPENAI / "env"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir: { WATCH_DIR }")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Data dir:    {WATCH_DATA_DIR}")
for quarter, path in QUARTER_FILES.items():
    print(f"{quarter}: {path}")


Project dir: /srv/notebooks/[6.0] ML Pricers/Base_OpenAI/watch
Output dir:  /srv/notebooks/[6.0] ML Pricers/Base_OpenAI/watch/outputs
Data dir:    /srv/notebooks/[6.0] ML Pricers/Data/watch
2025Q1: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q1_dedup_lot_url.csv
2025Q2: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q2_dedup_lot_url.csv
2025Q3: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q3_dedup_lot_url.csv
2025Q4: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q4_dedup_lot_url.csv


## Imports


In [2]:
import pandas as pd

from openai_pricer import load_explicit_quarter_files, prepare_pricing_frame
from openai_pricer.runner import run_asset_pricer
from watch_prompt_builder import build_prompt, build_system_prompt


## Load Quarter Data


In [3]:
raw_df = load_explicit_quarter_files(QUARTER_FILES, verbose=True)
pricing_df = prepare_pricing_frame(
    raw_df,
    rows_per_quarter=ROWS_PER_QUARTER,
    max_rows=MAX_ROWS,
    random_state=RANDOM_STATE,
    dedupe_key_col="lot_url",
    verbose=True,
)
print("Rows selected for pricing:", len(pricing_df))
print(pricing_df["source_quarter"].value_counts().sort_index())
pricing_df[[c for c in ['ID', 'source_quarter', 'sale_date', 'brand', 'model', 'ref_no', 'box', 'papers', 'ah_low_estimate_usd', 'ah_high_estimate_usd', 'ah_price_usd'] if c in pricing_df.columns]].head()


[openai data] loaded 2025Q1 watch_sales_2025Q1_dedup_lot_url.csv: 574 rows
[openai data] loaded 2025Q2 watch_sales_2025Q2_dedup_lot_url.csv: 1,681 rows
[openai data] loaded 2025Q3 watch_sales_2025Q3_dedup_lot_url.csv: 483 rows
[openai data] loaded 2025Q4 watch_sales_2025Q4_dedup_lot_url.csv: 2,257 rows
[openai data] combined quarter rows: 4,995
[openai data] quarters: {'2025Q1': 574, '2025Q2': 1681, '2025Q3': 483, '2025Q4': 2257}
[openai data] deduplicated lot_url: 4,995->4,995
Rows selected for pricing: 4995
source_quarter
2025Q1     574
2025Q2    1681
2025Q3     483
2025Q4    2257
Name: count, dtype: int64


,ID,source_quarter,sale_date,brand,model,ref_no,box,papers,ah_low_estimate_usd,ah_high_estimate_usd,ah_price_usd
0,9e21ed23-a799-4c91-8062-dda4f02dd4da,2025Q1,2025-02-19,Richard Mille,RM11-03TI,RM11-03TI,1.0,1.0,1300000.0,2000000.0,1167500.00
1,0973e57a-c9c3-4f97-b386-b73cdf9573f4,2025Q1,2025-02-19,Panerai,Luminor 1950 Tourbillon GMT Titanio,PAM00578,1.0,1.0,220000.0,440000.0,341333.33
2,a50617c3-7fe9-40a8-ae0e-fc98cd45c77a,2025Q1,2025-02-19,Audemars Piguet,Royal Oak,14882TT,1.0,0.0,130000.0,260000.0,447833.33
3,a1d98706-c98b-47aa-bf0d-dc6496c235f9,2025Q1,2025-02-19,Rolex,Explorer MK 3,1016,0.0,0.0,80000.0,160000.0,80000.00
4,3f05d203-d533-4516-b0c2-cb5ba0253349,2025Q1,2025-02-19,Piaget,Emperador Coussin XL Grande Lune Planisphère,P10764,1.0,1.0,180000.0,360000.0,256000.00


## Preview Prompt

The prompt excludes realized sale prices and existing model prediction columns.


In [4]:
system_prompt = build_system_prompt(INCLUDE_AUCTION_ESTIMATES)
example_prompt = build_prompt(
    pricing_df.iloc[0].to_dict(),
    INCLUDE_AUCTION_ESTIMATES,
    INCLUDE_MARKET_INDICATORS,
)
print("SYSTEM PROMPT:")
print(system_prompt[:2000])
print("\nUSER PROMPT:")
print(example_prompt[:4000])


SYSTEM PROMPT:
You are an expert luxury watch appraiser. Price the watch's expected sale value in USD under normal auction-market conditions. Use only the watch and auction metadata provided in the user prompt. Do not use realized sale price, buyer-premium price, or other model prediction columns as inputs. Do not use auction-house estimate fields for valuation even if they appear in metadata. Consider brand, model, reference number, production period, case size, case and band material, condition, box and papers, precious metals, diamonds, sale venue, location, sale date, and available market indicators. Return a realistic nonzero USD valuation. Respond only as JSON matching the requested schema.

USER PROMPT:
Price this luxury watch using the available fields below. Do not use any realized sale price or model prediction as an input.

Watch data:
Asset Type: Watch
Brand: Richard Mille
Model: RM11-03TI
Ref No: RM11-03TI
Case Size: 42 x 50
Case Material: titanium
Band Material: rubber/ti

## Run OpenAI Pricing

Requires `OPENAI_API_KEY` in `Base_OpenAI/env` or the environment. Keep `ROWS_PER_QUARTER=10` for a smoke test; set it to `None` for a full run.


In [5]:
run_asset_pricer(
    asset_name="watch",
    quarter_files=QUARTER_FILES,
    output_dir=OUTPUT_DIR,
    output_prefix=f"openai_gpt41mini_watch_{VARIANT}",
    model=MODEL,
    model_label=f"OpenAI GPT-4.1 Mini Watch - {'With Estimates' if INCLUDE_AUCTION_ESTIMATES else 'No Estimates'}",
    build_prompt=build_prompt,
    build_system_prompt=build_system_prompt,
    actual_col=TARGET_COL,
    include_auction_estimates=INCLUDE_AUCTION_ESTIMATES,
    include_market_indicators=INCLUDE_MARKET_INDICATORS,
    max_rows=MAX_ROWS,
    rows_per_quarter=ROWS_PER_QUARTER,
    random_state=RANDOM_STATE,
    resume=RESUME,
    checkpoint_every=CHECKPOINT_EVERY,
    max_workers=MAX_WORKERS,
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    max_retries=MAX_RETRIES,
    env_file=ENV_FILE,
)


[openai watch] output_prefix=openai_gpt41mini_watch_no_estimates, model=gpt-4.1-mini, include_auction_estimates=False, include_market_indicators=True, workers=50
[openai watch] predictions=/srv/notebooks/[6.0] ML Pricers/Base_OpenAI/watch/outputs/openai_gpt41mini_watch_no_estimates_predictions.csv
[openai data] loaded 2025Q1 watch_sales_2025Q1_dedup_lot_url.csv: 574 rows
[openai data] loaded 2025Q2 watch_sales_2025Q2_dedup_lot_url.csv: 1,681 rows
[openai data] loaded 2025Q3 watch_sales_2025Q3_dedup_lot_url.csv: 483 rows
[openai data] loaded 2025Q4 watch_sales_2025Q4_dedup_lot_url.csv: 2,257 rows
[openai data] combined quarter rows: 4,995
[openai data] quarters: {'2025Q1': 574, '2025Q2': 1681, '2025Q3': 483, '2025Q4': 2257}
[openai data] deduplicated lot_url: 4,995->4,995
[openai watch] rows selected: 4,995
source_quarter
2025Q1     574
2025Q2    1681
2025Q3     483
2025Q4    2257
[openai watch] resume found successful rows: 4,995
[openai watch] pending rows: 0
[openai run] checkpoint w

In [ ]:
print("SUCCESS")